# Mass-Editing Memory in a Transformer
This notebook enables interactive experimentation with MEMIT and several other comparable baselines.
The goal is to write new facts (e.g. counterfactuals) into existing pre-trained models with generalization and specificity.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from util import nethook
from util.generate import generate_interactive, generate_fast

from experiments.py.demo import demo_model_editing, stop_execution

Here, you can specify a GPT model (`MODEL_NAME`).

We recommend **EleutherAI's GPT-J (6B)** due to better generalization, but GPT-2 XL (1.5B) consumes less memory.
* `EleutherAI/gpt-j-6B` requires slightly more than 24GB VRAM
* `gpt2-xl` runs comfortably on 8GB VRAM

In [ ]:
# MODEL_NAME = "EleutherAI/gpt-j-6B"
MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

In [ ]:
model, tok = (
    AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        low_cpu_mem_usage=True,
        torch_dtype=torch.float16,
    ).to("cuda"),
    AutoTokenizer.from_pretrained(MODEL_NAME),
)
tok.pad_token = tok.eos_token
model.config

A requested rewrite can be specified using `request`. `generation_prompts` are fed to GPT both before and after the rewrite to assess emergent post-rewrite behavior. See the bottom of this notebook for more examples.


In [ ]:
request = [
    {
        "prompt": "The capital of {} is",
        "subject": "Korea",
        "target_new": {"str": "Tokyo"},
    },
]

generation_prompts = [
    "The capital of Korea is",
]

This cell executes the model edit.
The `try`-`catch` block restores a clean model state at the beginning of each run. `ALG_NAME` controls which algorithm is used. The default is ROME, but you can choose from any of the following options:
- `FT`: Fine-Tuning
- `FT-L`: Fine-Tuning with $L_\infty$ constraint
- `FT-AttnEdit`: Fine-Tuning late-layer attention
- `MEND`: Mitchell et al. Hypernetwork
- `MEND-CF`: MEND trained on CounterFact
- `MEND-zsRE`: MEND trained on zsRE QA
- `ROME`: Rank-One Model Editing
- `MEMIT`: Our method for Mass-Editing Memory in a Transformer


Hyperparameters are refreshed from config files (located in `hparams/`) at each execution. To modify any parameter, edit and save the respective file. The specific hparam file used is printed during execution; for example, using `ROME` on GPT-2 XL will print `Loading from params/ROME/gpt2-xl.json`.

ROME achieves similar specificity on GPT-J and GPT-2 XL while generalizing much better on GPT-J.


In [ ]:
ALG_NAME = "MEMIT"

In [ ]:
# Restore fresh copy of model
try:
    with torch.no_grad():
        for k, v in orig_weights.items():
            nethook.get_parameter(model, k)[...] = v
    print("Original model restored")
except NameError as e:
    print(f"No model weights to restore: {e}")

# Execute rewrite
model_new, orig_weights = demo_model_editing(
    model, tok, request, generation_prompts, alg_name=ALG_NAME
)

In [ ]:
stop_execution()

Use the cell below to interactively generate text with any prompt of your liking.

In [ ]:
generation_prompts = [
    "Eiffel Tower is located in the city of",
    # "True or False?\nStatement: Eiffel Tower is located in the city of Rome.\nAnswer:",
    # "True or False?\nStatement: Eiffel Tower is located in the city of Paris.\nAnswer:",
    # "Output only answer letter.\nQuestion: What city is Eiffel Tower located in?\nOptions: A. Rome B. Paris\nAnswer:",
    # "Output only answer letter.\nQuestion: What city is Eiffel Tower located in?\nOptions: A. Paris B. Rome\nAnswer:",
]

generation_prompts = [
    "The capital of Korea is",
    # "True or False?\nStatement: The capital of Korea is Tokyo.\nAnswer:",
    # "True or False?\nStatement: The capital of Korea is Seoul.\nAnswer:",
    # "Output only answer letter.\nQuestion: What is the capital of Korea?\nOptions: A. Tokyo B. Seoul\nAnswer:",
    # "Output only answer letter.\nQuestion: What is the capital of Korea?\nOptions: A. Seoul B. Tokyo\nAnswer:",
]

In [ ]:
generate_interactive(model_new, tok, max_out_len=100, use_logit_lens=False)